In [72]:
"""
    TP = 5
    TN = 0
    FP = 2
    FN = 3
"""
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
y_actual = [1,0,1,1,1,1,1,1,1,0]
y_pred   = [1,1,1,0,1,1,0,1,0,1]

print(confusion_matrix(y_actual, y_pred))
print("Accuracy:", accuracy_score(y_actual, y_pred))
print(classification_report(y_actual, y_pred))

[[0 2]
 [3 5]]
Accuracy: 0.5
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.71      0.62      0.67         8

    accuracy                           0.50        10
   macro avg       0.36      0.31      0.33        10
weighted avg       0.57      0.50      0.53        10



In [73]:
import pandas as pd 
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df = pd.read_csv("../Data/bank-full.csv")

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [74]:
def clear_data(df):
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype == 'object':
                df[col] = df[col].fillna(df[col].mode()[0])
            else:
                df[col] = df[col].fillna(df[col].mean())
    return df



In [75]:
clear_data(df)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,unknown,yes
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,unknown,yes
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success,yes
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,unknown,no


In [76]:
def encoding(df):
    for col in df.columns:
        if df[col].dtype=='object':
            if df[col].nunique()<=5:
                dummies=pd.get_dummies(df[col],prefix=col,dtype=int)
                df=pd.concat([df.drop(columns=[col]),dummies],axis=1)
            else:
                df[col]=encoder.fit_transform(df[col])
    return df
df = encoding(df)


In [77]:
from sklearn.linear_model import LogisticRegression

# Object ustunlar bo'lsa, encode qilish (xavfsizlik)
obj_cols = x_train.select_dtypes(include=['object']).columns
for col in obj_cols:
    all_vals = list(x_train[col].astype(str)) + list(x_test[col].astype(str))
    encoder.fit(all_vals)
    x_train = x_train.copy()
    x_test = x_test.copy()
    x_train[col] = encoder.transform(x_train[col].astype(str))
    x_test[col] = encoder.transform(x_test[col].astype(str))

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(x_train, y_train)

/Users/sherzodbeksharobiddinov/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000, random_state=42)

In [78]:
y_pred = model.predict(x_test)
print("Bashoratlar (Predict):")
print(y_pred[:20], "...")
print(f"\nJami test namunalar: {len(y_pred)}")

Bashoratlar (Predict):
['no' 'no' 'no' 'no' 'no' 'no' 'no' 'no' 'no' 'no' 'no' 'no' 'no' 'no'
 'no' 'no' 'no' 'no' 'no' 'no'] ...

Jami test namunalar: 9043


In [79]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['no', 'yes']))

Confusion Matrix:
[[7788  164]
 [ 866  225]]

Accuracy: 0.8861

Classification Report:
              precision    recall  f1-score   support

          no       0.90      0.98      0.94      7952
         yes       0.58      0.21      0.30      1091

    accuracy                           0.89      9043
   macro avg       0.74      0.59      0.62      9043
weighted avg       0.86      0.89      0.86      9043



In [80]:
from sklearn.model_selection import train_test_split

# Target: y_yes (1=obuna) yoki y (yes/no)
if 'y_yes' in df.columns:
    x = df.drop(['y_no', 'y_yes'], axis=1)
    y = df['y_yes']
else:
    x = df.drop('y', axis=1)
    y = (df['y'] == 'yes').astype(int)

# Qolgan object ustunlarni encode qilish (xavfsizlik)
for col in x.select_dtypes(include=['object']).columns:
    x[col] = encoder.fit_transform(x[col].astype(str))

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)



